# 03 — LCEL: Chaining Runnables Together
### `RunnableLambda`, `RunnableSequence`, `RunnableParallel`

Every piece we used in notebooks 1-2 — prompts, models, parsers — is a
**`Runnable`**. That's the actual reason `prompt | model | parser` works:
`|` is operator overloading that builds a `RunnableSequence` out of any two
Runnables. This notebook covers the composition layer itself, which is what
`pipeline.py` leans on to orchestrate the whole SupportPilot ticket flow.

## 3.1 Everything is a Runnable

A `Runnable` is anything with `.invoke()`, `.batch()`, and `.stream()`.
Prompts, models, parsers, plain Python functions (wrapped), and entire
chains all satisfy this interface — which is *why* they can be composed
with each other interchangeably.

In [ ]:
from langchain_core.runnables import RunnableLambda

# The simplest possible Runnable: wrap a plain Python function.
double = RunnableLambda(lambda x: x * 2)
print(double.invoke(21))

# .batch() runs it over a list of inputs
print(double.batch([1, 2, 3]))


This is exactly the mechanism `chains.py` uses for **mock mode** — instead of
a real `prompt | model | parser` chain, each mock chain is just a
`RunnableLambda` wrapping an offline heuristic function:

```python
def build_classification_chain():
    if MODE == "mock":
        return RunnableLambda(lambda inputs: mock.mock_classify(inputs["ticket_id"], inputs["ticket_text"]))
    ...
```

The rest of `pipeline.py` calls `.invoke()` on whatever `build_classification_chain()`
returns — it never needs to know whether that's a fake function or a real
model call. That's the entire point of the Runnable interface.

In [ ]:
import sys
sys.path.insert(0, ".")   # make sure the project root is importable
import mock_heuristics as mock
from langchain_core.runnables import RunnableLambda

classification_chain = RunnableLambda(
    lambda inputs: mock.mock_classify(inputs["ticket_id"], inputs["ticket_text"])
)

result = classification_chain.invoke({
    "ticket_id": "T001",
    "ticket_text": "Where is my order? It's been 6 days.",
})
print(result)


## 3.2 `RunnableSequence` — the `|` operator

You've already used this without naming it: `prompt | model` builds a
`RunnableSequence`. Each step's output becomes the next step's input.

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Draft a short, friendly reply."),
    ("human", "{ticket_text}"),
])
fake_model = FakeListChatModel(responses=["Thanks for reaching out, we'll look into it right away."])

draft_chain = prompt | fake_model | StrOutputParser()
print(draft_chain.invoke({"ticket_text": "Where is my order?"}))

# You can inspect the sequence's steps directly:
print()
print(draft_chain.steps)


## 3.3 `RunnableParallel` — running independent steps concurrently

Two steps in the SupportPilot pipeline don't depend on each other's output —
**account lookup** and **knowledge retrieval** both only need the
classification result, not each other. `RunnableParallel` expresses that
explicitly, rather than hiding the independence inside sequential code.

In [ ]:
from langchain_core.runnables import RunnableParallel
import time

def slow_lookup(x):
    time.sleep(0.3)
    return f"account info for {x['customer_id']}"

def slow_retrieve(x):
    time.sleep(0.3)
    return f"kb results for '{x['query']}'"

context_gathering = RunnableParallel(
    customer_ctx=RunnableLambda(slow_lookup),
    kb_chunks=RunnableLambda(slow_retrieve),
)

start = time.time()
result = context_gathering.invoke({"customer_id": "CUST001", "query": "refund"})
elapsed = time.time() - start

print(result)
print(f"elapsed: {elapsed:.2f}s")  # ~0.3s, not ~0.6s -- both branches ran concurrently


Compare this to the real thing in `pipeline.py`:

```python
_context_gathering = RunnableParallel(
    customer_ctx=RunnableLambda(lambda x: _account_lookup(x["resolved_customer_id"])),
    kb_chunks=RunnableLambda(lambda x: _knowledge_retrieval(x["classification"])),
)
```

Same shape — two independent lookups, run together, results collected into
one dict with named keys (`customer_ctx`, `kb_chunks`) that the next step
(drafting) reads by name.

## 3.4 Why the escalation step is *not* a Runnable

Every step so far has been chainable. `pipeline.py`'s escalation decision
deliberately is **not**:

```python
decision = escalation_rules.decide_escalation(
    issue_type=classification["issue_type"], ...
)
```

This is a plain function call, not a Runnable in the chain. That's on
purpose — see `escalation_rules.py`'s docstring. The hard-category
escalation gate (fraud, high-value refunds, "get me a manager") has to be
code a model can't influence, which means it can't live inside anything a
prompt touches. LCEL is great for composing model-facing steps; it's the
wrong tool for a safety-critical business rule that must never depend on
what a model decided.

## Exercise

1. Build a `RunnableParallel` with three branches instead of two — add a
   fake "prior ticket count" lookup alongside `customer_ctx` and `kb_chunks`.
2. Chain the output of that `RunnableParallel` into a `RunnableLambda` that
   combines all three branches into a single formatted string. (Hint: the
   parallel step's output is a dict — your lambda's input is that dict.)
3. Look at `pipeline.py`'s `run_ticket()` end to end and list, in order,
   which parts are LCEL Runnables and which parts are plain Python function
   calls. What's the pattern in *which* steps end up as Runnables?